In [ ]:
import json
import pandas as pd
from pathlib import Path

# Путь к папке с данными
raw_dir = Path("data/raw/variant_10")

# Находим все JSON-файлы в папке
json_files = list(raw_dir.glob("*.json"))

if not json_files:
    raise FileNotFoundError(f"В папке {raw_dir} нет JSON-файлов")

# Выбираем самый свежий (по дате изменения)
latest_file = max(json_files, key=lambda f: f.stat().st_mtime)
print(f"[INFO] Выбран файл: {latest_file.name}")

# Открываем и читаем JSON
with open(latest_file, 'r', encoding='utf-8') as f:
    raw_data = json.load(f)

print("\n" + "=" * 60)
print("ИЗУЧЕНИЕ СТРУКТУРЫ JSON")
print("=" * 60)

print("Тип данных:", type(raw_data))

if isinstance(raw_data, list):
    print("Длина списка:", len(raw_data))
    print("Тип первого элемента:", type(raw_data[0]))
    
    if len(raw_data) > 1 and isinstance(raw_data[1], list):
        print("Количество записей:", len(raw_data[1]))
        print("\nПервая запись (пример):")
        first_record = raw_data[1][0]
        for key, value in first_record.items():
            print(f"  {key}: {value}")

In [ ]:
# Берём записи (второй элемент списка)
records = raw_data[1]

# Создаём список простых словарей
normalized_records = []
for item in records:
    normalized_records.append({
        'year': int(item['date']),
        'value': item['value'],
        'country_iso3': item['country']['id'],  # 'DE'
        'country_name': item['country']['value'],  # 'Germany'
        'indicator_code': item['indicator']['id'],
        'indicator_name': item['indicator']['value']
    })

df = pd.DataFrame(normalized_records)

print("Первые 5 строк:")
df.head()

In [ ]:
print("Форма таблицы:", df.shape)  # (сколько строк, сколько столбцов)
print("\nКолонки:", df.columns.tolist())
print("\nТипы данных:")
df.dtypes
print("\nПервые 5 строк:")
df.head()
print("\nПоследние 5 строк:")
df.tail()

In [ ]:
# year уже int, но убедимся
df['year'] = pd.to_numeric(df['year'], errors='coerce')

# value превращаем в число (пропуски станут NaN)
df['value'] = pd.to_numeric(df['value'], errors='coerce')

print("Типы после приведения:")
df.dtypes

In [ ]:
print("Пропуски до:")
print(df.isnull().sum())

# Удалим строки, где нет года (хотя такого быть не должно)
df = df.dropna(subset=['year'])

print("\nПропуски после:")
print(df.isnull().sum())

In [ ]:
# Проверим дубликаты
duplicates = df.duplicated(subset=['year', 'country_iso3', 'indicator_code']).sum()
print(f"Найдено дубликатов: {duplicates}")

if duplicates > 0:
    df = df.drop_duplicates(subset=['year', 'country_iso3', 'indicator_code'])
    print("Дубликаты удалены")

In [ ]:
df = df.sort_values('year').reset_index(drop=True)
print("Первые 5 строк после сортировки:")
df.head()

In [ ]:
print("Форма после очистки:", df.shape)
print("\nТипы после очистки:")
df.dtypes
print("\nПервые 5 строк:")
df.head()
print("\nСтатистика по value:")
df['value'].describe()

In [ ]:
from datetime import datetime

# Создаём папку, если её нет
norm_dir = Path("data/normalized/variant_10")
norm_dir.mkdir(parents=True, exist_ok=True)

# Имя файла с временной меткой
timestamp = datetime.now().strftime("%Y-%m-%d_%H-%M-%S")
csv_path = norm_dir / f"{timestamp}.csv"

# Сохраняем
df.to_csv(csv_path, index=False, encoding='utf-8')
print(f"Сохранено в {csv_path}")

In [ ]:
# Проверим, что файл создался
if csv_path.exists():
    print(f" Файл успешно создан: {csv_path}")
    print(f"Размер файла: {csv_path.stat().st_size} байт")
else:
    print(" Файл не найден")